In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import math
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import datetime
pd.set_option('display.max_columns', None) 

# Clean raw data

## Delist

In [3]:
delist = pd.read_csv('delisting14-24.csv')

In [9]:
delist.columns

Index(['PERMNO', 'DelistingDt', 'DelDtPrc', 'DelDtPrcFlg', 'DelActionType',
       'DelStatusType', 'DelReasonType', 'DelPaymentType', 'DelPERMNO',
       'DelPERMCO', 'DelRet', 'DelRetMissType', 'DelNextDt', 'DelNextPrc',
       'DelNextPrcFlg', 'DelAmtDt', 'DelDivAmt', 'DelDisType', 'DelDlyDt',
       'PrimaryExch', 'SICCD', 'NASDIssuno'],
      dtype='object')

In [5]:
# Filter for bankruptcy delistings
bkpy = delist[delist['DelReasonType']=='BKPY'].copy()
# Convert DelistingDt to datetime and extract year
bkpy['DelistingDt'] = pd.to_datetime(bkpy['DelistingDt'], errors='coerce')
bkpy['DelistingDt_year']= bkpy['DelistingDt'].dt.year #轉換 DelistingDt 列為日期格式且提取年份
# Drop rows with invalid dates
bkpy = bkpy[(bkpy['DelistingDt_year']>=2019) & (bkpy['DelistingDt_year']<=2024)]

In [27]:
bkpy.shape

(102, 23)

In [29]:
duplicates = bkpy[bkpy['PERMNO'].duplicated(keep=False)]

In [31]:
duplicates

,PERMNO,DelistingDt,DelDtPrc,DelDtPrcFlg,DelActionType,DelStatusType,DelReasonType,DelPaymentType,DelPERMNO,DelPERMCO,DelRet,DelRetMissType,DelNextDt,DelNextPrc,DelNextPrcFlg,DelAmtDt,DelDivAmt,DelDisType,DelDlyDt,PrimaryExch,SICCD,NASDIssuno,DelistingDt_year


In [11]:
link = pd.read_excel('link table.xlsx') # provide gvkey

In [13]:
link.columns

Index(['Global Company Key', 'Company Name', 'Ticker Symbol', 'CUSIP',
       'CIK Number', 'Standard Industry Classification Code',
       'North American Industry Classification Code', 'Primary Link Marker',
       'Security-level Identifier', 'Link Type Codd',
       'Historical CRSP PERMNO Link to COMPUSTAT Record',
       'Historical CRSP PERMCO Link to COMPUSTAT Record',
       'First Effective Date of Link', 'Last Effective Date of Link'],
      dtype='object')

In [33]:
duplicates2 = link[link['Historical CRSP PERMNO Link to COMPUSTAT Record'].duplicated(keep=False)]
duplicates2

,Global Company Key,Company Name,Ticker Symbol,CUSIP,CIK Number,Standard Industry Classification Code,North American Industry Classification Code,Primary Link Marker,Security-level Identifier,Link Type Codd,Historical CRSP PERMNO Link to COMPUSTAT Record,Historical CRSP PERMCO Link to COMPUSTAT Record,First Effective Date of Link,Last Effective Date of Link
6,1007,ABKCO INDUSTRIES INC,4135B,000774109,1882.0,3652,334612.0,P,01,LU,10058,20,1979-01-31,1984-09-28
7,1007,ABKCO INDUSTRIES INC,4135B,000774109,1882.0,3652,334612.0,C,00X,LU,10058,20,1973-10-01,1979-01-30
8,1008,ABM COMPUTER SYSTEMS INC,ABMC.,000775106,NaN,3577,334119.0,P,01,LC,10066,6331,1983-08-25,1987-02-26
10,1010,ACF INDUSTRIES INC,4165A,00099V004,910627.0,3743,336510.0,P,01,LU,10006,22156,1962-01-31,1984-06-28
11,1010,ACF INDUSTRIES INC,4165A,00099V004,910627.0,3743,336510.0,C,00X,LU,10006,22156,1950-05-01,1962-01-30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32382,345556,F-STAR THERAPEUTICS INC,FSTX,30315R107,1566373.0,2836,325414.0,P,01,LC,16069,55573,2020-11-23,2023-03-31
32389,347007,IMMUNITYBIO INC,IBRX,45256X103,1326110.0,2836,325414.0,P,01,LC,15533,55364,2021-03-10,NaT
32396,349530,NEXTPLAY TECHNOLOGIES INC,NXTP,65344G201,1372183.0,7310,541810.0,P,01,LC,17324,56274,2021-07-09,2024-04-12
32398,349972,INDAPTUS THERAPEUTICS INC,INDP,45339J204,1857044.0,2836,325414.0,P,01,LC,15642,55395,2021-08-04,NaT


In [15]:
# Merge bkpy with link table on PERMNO and rename columns

merge_delist_link = pd.merge(bkpy, link, left_on='PERMNO', right_on='Historical CRSP PERMNO Link to COMPUSTAT Record', how='left')
merge_delist_link.rename(columns={'DelistingDt':'delist date',
                                  'DelistingDt_year':'delist year',
                                  'Global Company Key':'gvkey',
                                  'Ticker Symbol':'ticker'}, inplace=True)
merge_delist_link

,PERMNO,delist date,DelDtPrc,DelDtPrcFlg,DelActionType,DelStatusType,DelReasonType,DelPaymentType,DelPERMNO,DelPERMCO,DelRet,DelRetMissType,DelNextDt,DelNextPrc,DelNextPrcFlg,DelAmtDt,DelDivAmt,DelDisType,DelDlyDt,PrimaryExch,SICCD,NASDIssuno,delist year,gvkey,Company Name,ticker,CUSIP,CIK Number,Standard Industry Classification Code,North American Industry Classification Code,Primary Link Marker,Security-level Identifier,Link Type Codd,Historical CRSP PERMNO Link to COMPUSTAT Record,Historical CRSP PERMCO Link to COMPUSTAT Record,First Effective Date of Link,Last Effective Date of Link
0,10180,2020-05-29,0.0900,TR,GDR,VCL,BKPY,PRCF,0,0,-0.166667,NaN,2020-06-01,0.075,DP,2020-06-01,0.0,NO,2020-06-01,Q,2834,12980,2020,14304.0,AKORN OPERATING COMPANY LLC,AKRXQ,009728106,3116.0,2834.0,325412.0,P,01,LU,10180.0,9743.0,1988-04-08,2020-10-05
1,10656,2019-04-02,0.1345,TR,GDR,VCL,BKPY,PRCF,0,0,-0.182156,NaN,2019-04-03,0.110,DP,2019-04-03,0.0,NO,2019-04-03,Q,5160,54,2019,1094.0,ACETO CORP,ACETQ,004446100,2034.0,5160.0,424690.0,P,01,LC,10656.0,37.0,1972-12-14,2019-10-02
2,12660,2020-06-29,0.5500,TR,GDR,VCL,BKPY,PRCF,0,0,-0.336364,NaN,2020-07-01,0.365,DP,2020-07-01,0.0,NO,2020-06-30,N,5999,0,2020,185645.0,GNC HOLDINGS INC,GNCIQ,36191G107,1502034.0,5400.0,446191.0,P,01,LC,12660.0,53710.0,2011-04-01,2020-10-31
3,12792,2020-08-03,1.9000,TR,GDR,VCL,BKPY,PRCF,0,0,-0.705263,NaN,2020-08-04,0.560,DP,2020-08-04,0.0,NO,2020-08-04,Q,9999,68308,2020,187215.0,GLOBAL EAGLE ENTERTAINMENT,GEENQ,37951D300,1512077.0,4899.0,517410.0,P,02,LC,12792.0,53773.0,2011-05-27,2021-03-31
4,12796,2024-11-15,1.0800,TR,GDR,VCL,BKPY,PRCF,0,0,-0.861111,NaN,2024-11-19,0.150,DP,2024-11-19,0.0,NO,2024-11-18,N,4512,67349,2024,185624.0,SPIRIT AVIATION HOLDINGS INC,FLYY,84863V101,1498710.0,4512.0,481111.0,P,01,LC,12796.0,53776.0,2011-05-31,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124,91739,2019-06-27,0.0395,TR,GDR,VCL,BKPY,PRCF,0,0,-0.417722,NaN,2019-06-28,0.023,DP,2019-06-28,0.0,NO,2019-06-28,Q,1311,54702,2019,166436.0,LEGACY RESERVES INC,LGCYQ,524706108,1735828.0,1311.0,211120.0,P,01,LC,91739.0,51657.0,2007-01-12,2019-12-13
125,92400,2024-08-09,0.8438,TR,GDR,VCL,BKPY,PRCF,0,0,-0.869637,NaN,2024-08-13,0.110,DP,2024-08-13,0.0,NO,2024-08-12,N,5031,0,2024,178806.0,LL FLOORING HOLDINGS INC,LLFLQ,55003T107,1396033.0,5211.0,444110.0,P,01,LC,92400.0,52839.0,2007-11-09,2024-08-09
126,93420,2020-10-09,0.1550,TR,GDR,VCL,BKPY,PRCF,0,0,-0.354839,NaN,2020-10-12,0.100,DP,2020-10-12,0.0,NO,2020-10-12,Q,1321,66380,2020,184442.0,CHORD ENERGY CORP,CHRD,674215207,1486159.0,1311.0,2111.0,N,01,LC,93420.0,53438.0,2020-11-20,2020-11-30
127,93420,2020-10-09,0.1550,TR,GDR,VCL,BKPY,PRCF,0,0,-0.354839,NaN,2020-10-12,0.100,DP,2020-10-12,0.0,NO,2020-10-12,Q,1321,66380,2020,184442.0,CHORD ENERGY CORP,CHRD,674215207,1486159.0,1311.0,2111.0,C,01,LC,93420.0,53438.0,2020-11-01,2020-11-19


In [17]:
merge_delist_link.shape

(129, 37)

In [19]:
#列出所有重複 PERMNO的 row
duplicates_in_column = merge_delist_link[merge_delist_link['PERMNO'].duplicated(keep=False)]
duplicates_in_column

,PERMNO,delist date,DelDtPrc,DelDtPrcFlg,DelActionType,DelStatusType,DelReasonType,DelPaymentType,DelPERMNO,DelPERMCO,DelRet,DelRetMissType,DelNextDt,DelNextPrc,DelNextPrcFlg,DelAmtDt,DelDivAmt,DelDisType,DelDlyDt,PrimaryExch,SICCD,NASDIssuno,delist year,gvkey,Company Name,ticker,CUSIP,CIK Number,Standard Industry Classification Code,North American Industry Classification Code,Primary Link Marker,Security-level Identifier,Link Type Codd,Historical CRSP PERMNO Link to COMPUSTAT Record,Historical CRSP PERMCO Link to COMPUSTAT Record,First Effective Date of Link,Last Effective Date of Link
6,13113,2020-06-26,0.2590,TR,GDR,VCL,BKPY,PRCF,0,0,-0.795367,NaN,2020-06-30,0.053,DP,2020-06-30,0.0,NO,2020-06-29,A,1311,63247,2020,186061.0,LILIS ENERGY INC,LLEXQ,532403201,1437557.0,1311.0,211120.0,P,01,LC,13113.0,53936.0,2017-03-14,2020-12-07
7,13113,2020-06-26,0.2590,TR,GDR,VCL,BKPY,PRCF,0,0,-0.795367,NaN,2020-06-30,0.053,DP,2020-06-30,0.0,NO,2020-06-29,A,1311,63247,2020,186061.0,LILIS ENERGY INC,LLEXQ,532403201,1437557.0,1311.0,211120.0,P,01,LC,13113.0,53936.0,2011-11-02,2016-05-25
10,13263,2020-01-10,0.5000,TR,GDR,VCL,BKPY,PRCF,0,0,-0.120000,NaN,2020-01-13,0.440,DP,2020-01-13,0.0,NO,2020-01-13,Q,9999,69245,2020,190580.0,CEMPRA INC,CEMP,15130J109,NaN,2834.0,325412.0,P,01,LC,13263.0,53995.0,2012-02-03,2017-11-05
11,13263,2020-01-10,0.5000,TR,GDR,VCL,BKPY,PRCF,0,0,-0.120000,NaN,2020-01-13,0.440,DP,2020-01-13,0.0,NO,2020-01-13,Q,9999,69245,2020,196856.0,MELINTA THERAPEUTICS INC,MLNTQ,58549G209,1461993.0,2834.0,325412.0,P,01,LC,13263.0,53995.0,2017-11-06,2020-04-30
19,14756,2020-09-23,0.4300,TR,GDR,VCL,BKPY,PRCF,0,0,-0.781395,NaN,2020-09-24,0.094,DP,2020-09-24,0.0,NO,2020-09-24,N,6726,0,2020,317260.0,HERMITAGE OFFSHORE SER - OLD,NAO.3,G6599Y938,NaN,4412.0,483111.0,P,01,LC,14756.0,54928.0,2014-06-12,2019-04-17
20,14756,2020-09-23,0.4300,TR,GDR,VCL,BKPY,PRCF,0,0,-0.781395,NaN,2020-09-24,0.094,DP,2020-09-24,0.0,NO,2020-09-24,N,6726,0,2020,335466.0,HERMITAGE OFFSHORE SERVICES,HOFSQ,G4511M108,1597659.0,4400.0,532411.0,P,01,LC,14756.0,54928.0,2020-06-01,2021-06-30
21,14756,2020-09-23,0.4300,TR,GDR,VCL,BKPY,PRCF,0,0,-0.781395,NaN,2020-09-24,0.094,DP,2020-09-24,0.0,NO,2020-09-24,N,6726,0,2020,335466.0,HERMITAGE OFFSHORE SERVICES,HOFSQ,G4511M108,1597659.0,4400.0,532411.0,C,01,LC,14756.0,54928.0,2019-04-18,2020-05-31
27,15443,2023-11-28,0.3426,TR,GDR,VCL,BKPY,PRCF,0,0,-0.649737,NaN,2023-11-30,0.120,DP,2023-11-30,0.0,NO,2023-11-29,A,2834,0,2023,19722.0,BIOPHARMX CORP,BPMX,09072X309,NaN,2834.0,325412.0,P,01,LC,15443.0,55323.0,2015-06-30,2020-05-18
28,15443,2023-11-28,0.3426,TR,GDR,VCL,BKPY,PRCF,0,0,-0.649737,NaN,2023-11-30,0.120,DP,2023-11-30,0.0,NO,2023-11-29,A,2834,0,2023,36506.0,TREX WIND DOWN INC,TMBRQ,887080208,1504167.0,2834.0,325412.0,P,01,LC,15443.0,55323.0,2020-05-19,2024-05-09
32,15970,2020-07-31,0.3630,TR,GDR,VCL,BKPY,PRCF,0,0,-0.765840,NaN,2020-08-03,0.085,DP,2020-08-03,0.0,NO,2020-08-03,Q,9999,76845,2020,26825.0,KLR ENERGY ACQUISITION CORP,KLRE,49877M108,NaN,9995.0,999990.0,P,02,LC,15970.0,55535.0,2016-03-29,2017-04-27


In [23]:
duplicates_in_column.shape

(46, 37)

In [21]:
# del missing gvkey and duplicate PERMNO-gvkey pairs
merge_delist_link = merge_delist_link.dropna(subset=['gvkey']) #刪除'gvkey'欄位為缺失值（NaN）的行
merge_delist_link = merge_delist_link.drop_duplicates(subset=['PERMNO', 'gvkey'], keep='first') 
#判斷重複:有當兩行或更多行的 'PERMNO' 值和 'gvkey' 值都完全相同時，才被視為重複

In [93]:
#再看看還有哪些有重複的PERMNO
duplicates_in_column = merge_delist_link[merge_delist_link['PERMNO'].duplicated(keep=False)]
duplicates_in_column

,PERMNO,delist date,DelDtPrc,DelDtPrcFlg,DelActionType,DelStatusType,DelReasonType,DelPaymentType,DelPERMNO,DelPERMCO,DelRet,DelRetMissType,DelNextDt,DelNextPrc,DelNextPrcFlg,DelAmtDt,DelDivAmt,DelDisType,DelDlyDt,PrimaryExch,SICCD,NASDIssuno,delist year,gvkey,Company Name,ticker,CUSIP,CIK Number,Standard Industry Classification Code,North American Industry Classification Code,Primary Link Marker,Security-level Identifier,Link Type Codd,Historical CRSP PERMNO Link to COMPUSTAT Record,Historical CRSP PERMCO Link to COMPUSTAT Record,First Effective Date of Link,Last Effective Date of Link
10,13263,2020-01-10,0.5000,TR,GDR,VCL,BKPY,PRCF,0,0,-0.120000,NaN,2020-01-13,0.440,DP,2020-01-13,0.0,NO,2020-01-13,Q,9999,69245,2020,190580.0,CEMPRA INC,CEMP,15130J109,NaN,2834.0,325412.0,P,01,LC,13263.0,53995.0,2012-02-03,2017-11-05
11,13263,2020-01-10,0.5000,TR,GDR,VCL,BKPY,PRCF,0,0,-0.120000,NaN,2020-01-13,0.440,DP,2020-01-13,0.0,NO,2020-01-13,Q,9999,69245,2020,196856.0,MELINTA THERAPEUTICS INC,MLNTQ,58549G209,1461993.0,2834.0,325412.0,P,01,LC,13263.0,53995.0,2017-11-06,2020-04-30
19,14756,2020-09-23,0.4300,TR,GDR,VCL,BKPY,PRCF,0,0,-0.781395,NaN,2020-09-24,0.094,DP,2020-09-24,0.0,NO,2020-09-24,N,6726,0,2020,317260.0,HERMITAGE OFFSHORE SER - OLD,NAO.3,G6599Y938,NaN,4412.0,483111.0,P,01,LC,14756.0,54928.0,2014-06-12,2019-04-17
20,14756,2020-09-23,0.4300,TR,GDR,VCL,BKPY,PRCF,0,0,-0.781395,NaN,2020-09-24,0.094,DP,2020-09-24,0.0,NO,2020-09-24,N,6726,0,2020,335466.0,HERMITAGE OFFSHORE SERVICES,HOFSQ,G4511M108,1597659.0,4400.0,532411.0,P,01,LC,14756.0,54928.0,2020-06-01,2021-06-30
27,15443,2023-11-28,0.3426,TR,GDR,VCL,BKPY,PRCF,0,0,-0.649737,NaN,2023-11-30,0.120,DP,2023-11-30,0.0,NO,2023-11-29,A,2834,0,2023,19722.0,BIOPHARMX CORP,BPMX,09072X309,NaN,2834.0,325412.0,P,01,LC,15443.0,55323.0,2015-06-30,2020-05-18
28,15443,2023-11-28,0.3426,TR,GDR,VCL,BKPY,PRCF,0,0,-0.649737,NaN,2023-11-30,0.120,DP,2023-11-30,0.0,NO,2023-11-29,A,2834,0,2023,36506.0,TREX WIND DOWN INC,TMBRQ,887080208,1504167.0,2834.0,325412.0,P,01,LC,15443.0,55323.0,2020-05-19,2024-05-09
32,15970,2020-07-31,0.3630,TR,GDR,VCL,BKPY,PRCF,0,0,-0.765840,NaN,2020-08-03,0.085,DP,2020-08-03,0.0,NO,2020-08-03,Q,9999,76845,2020,26825.0,KLR ENERGY ACQUISITION CORP,KLRE,49877M108,NaN,9995.0,999990.0,P,02,LC,15970.0,55535.0,2016-03-29,2017-04-27
33,15970,2020-07-31,0.3630,TR,GDR,VCL,BKPY,PRCF,0,0,-0.765840,NaN,2020-08-03,0.085,DP,2020-08-03,0.0,NO,2020-08-03,Q,9999,76845,2020,31156.0,ROSEHILL RESOURCES INC,ROSEQ,777385105,1659122.0,1311.0,211120.0,P,02,LC,15970.0,55535.0,2017-05-01,2020-09-08
41,17090,2023-12-26,0.0489,TR,GDR,VCL,BKPY,PRCF,0,0,-0.754601,NaN,2023-12-27,0.012,DP,2023-12-27,0.0,NO,2023-12-27,A,6799,0,2023,32561.0,LEGACY ACQUISITION,LGC,524643103,NaN,9995.0,999990.0,P,01,LC,17090.0,56138.0,2017-12-01,2020-11-22
43,17090,2023-12-26,0.0489,TR,GDR,VCL,BKPY,PRCF,0,0,-0.754601,NaN,2023-12-27,0.012,DP,2023-12-27,0.0,NO,2023-12-27,A,6799,0,2023,37426.0,PARTS ID INC,IDICQ,702141102,1698113.0,5961.0,441330.0,P,01,LC,17090.0,56138.0,2020-11-23,2024-02-29


In [95]:
merge_delist_link.shape

(111, 37)

In [97]:
# Select relevant columns for final output
bkpy = merge_delist_link.loc[:,['PERMNO','gvkey','delist date','delist year']]
# Save
# bkpy.to_csv('/Users/mabel/Desktop/Paper Data/US Bankruptcy/EDA file/clean_delisting.csv', index=False)

In [99]:
bkpy.shape

(111, 4)

## Financial statement

In [101]:
finance = pd.read_csv('financial statement 10-k2018-2024.csv', low_memory=False)

In [103]:
# Filter for US companies with USD currency
fin = finance.loc[(finance.fic == 'USA') & (finance['curcd'] == 'USD') ].copy()

# Convert datadate to datetime
# fin['datadate']  = pd.to_datetime(fin['datadate'], format='mixed')

In [105]:
fin

,costat,curcd,datafmt,indfmt,consol,tic,datadate,gvkey,conm,cusip,cik,exchg,fyr,fic,sic,fyear,acco,acdo,aco,acodo,acominc,acox,acoxar,act,aedi,aldo,ao,aocidergl,aociother,aocipen,aocisecgl,aodo,aox,ap,apb,apc,apofs,arb,arc,artfs,at,bast,bkvlps,ca,caps,cb,ceq,ceql,ceqt,ch,che,chs,cld2,cld3,cld4,cld5,clfc,clfx,clg,clis,cll,cllc,clo,clrll,clt,cmp,crv,crvnli,cstk,cstkcv,dc,dclo,dcom,dcpstk,dcs,dcvsr,dcvsub,dcvt,dd,dd1,dd2,dd3,dd4,dd5,dfpac,dfs,dlc,dlto,dltp,dltsub,dltt,dm,dn,dpacb,dpacc,dpacli,dpacls,dpacme,dpacnr,dpaco,dpacre,dpact,dpdc,dpltb,dpsc,dpstb,dptb,dptc,dptic,dpvieb,dpvio,dpvir,drc,drci,drlt,ds,dudd,dvpa,dvpibb,dxd2,dxd3,dxd4,dxd5,ea,esopct,esopdlt,esopnr,esopr,esopt,excadj,fatb,fatc,fate,fatl,fatn,fato,fatp,fdfr,fea,fel,ffs,gdwl,geqrv,govgr,iaeq,iaeqci,iaeqmi,iafici,iafxi,iafxmi,iali,ialoi,ialti,iamli,iaoi,iapli,iarei,iasci,iasmi,iassi,iasti,iatci,iati,iatmi,iaui,icapt,intan,intano,invfg,invo,invofs,invreh,invrei,invres,invrm,invt,invwip,ip,ipc,ipv,iseq,iseqc,iseqm,isfi,isfxc,isfxm,islg,islgc,islgm,islt,isng,isngc,isngm,isotc,isoth,isotm,issc,issm,issu,ist,istc,istm,isut,itcb,ivaeq,ivao,ivgod,ivpt,ivst,lcabg,lcacl,lcacr,lcag,lcal,lcalt,lcam,lcao,lcast,lcat,lco,lcox,lcoxar,lcoxdr,lct,lcuacu,lif,lifr,lloml,lloo,llot,lo,loxdr,lrv,ls,lse,lt,mib,mrc1,mrc2,mrc3,mrc4,mrc5,mrct,mrcta,msa,msvrv,mtl,nat,np,npanl,npaore,nparl,npat,ob,optprcca,optprcex,optprcey,optprcgr,optprcwa,ppegt,ppenb,ppenc,ppenli,ppenls,ppenme,ppennr,ppeno,ppent,ppevbb,ppeveb,ppevo,ppevr,prc,prodv,prvt,pstk,pstkc,pstkl,pstkn,pstkr,pstkrv,pvcl,pvpl,pvt,radp,ragr,rari,rati,rcl,rdp,re,rea,reajo,recco,recd,rect,recta,rectr,recub,ret,reuna,reunr,rll,rlo,rlp,rlri,rlt,rpag,rreps,rvbci,rvbpi,rvbti,rvdo,rvdt,rveqt,rvlrv,rvno,rvnt,rvri,rvsi,rvti,rvtxr,rvupi,rvutx,saa,sal,sbdc,sc,sco,secu,seq,seqo,srt,ssnp,stbo,stio,tdscd,tdsce,tdslg,tdsmm,tdsng,tdso,tdss,tdst,tlcf,transa,tsa,tso,tstk,tstkc,tstkme,tstkp,txdb,txdba,txdbca,txdbcl,txditc,txndb,txndba,txndbl,txndbr,txp,txr,uaox,uapt,ucaps,uccons,uceq,ucustad,udcopres,udd,udmb,udolt,udpco,ui,uinvt,ulcm,ulco,unl,unnp,unnpl,uopres,upmcstk,upmpf,upmpfs,upmsubp,upstk,upstkc,upstksf,urect,urectr,urevub,usubpstk,vpac,vpo,wcap,xacc,xpp,acchg,adpac,am,amdc,amgw,aqa,aqd,aqeps,aqi,aqp,aqs,arce,arced,arceeps,autxr,balr,banlr,batr,bcef,bclr,bcltbl,bcnlr,bcrbl,bct,bctbl,bctr,bltbl,cbi,cdpac,cfbd,cfere,cfo,cfpdo,cga,cgri,cgti,cgui,cibegni,cicurr,cidergl,ciother,cipen,cisecgl,citotal,cnltbl,cogs,cpcbl,cpdoi,cpnli,cppbl,cprei,cstke,dbi,dfxa,diladj,dilavx,do,donr,dp,dpret,dtea,dted,dteeps,dtep,dvc,dvdnp,dvp,dvpd,dvpdp,dvrpiv,dvrre,dvsco,dvt,ebit,ebitda,eiea,emol,epsfi,epsfx,epspi,epspx,esub,fatd,fca,ffo,gbbl,gdwlam,gdwlia,gdwlid,gdwlieps,gdwlip,gla,glcea,glced,glceeps,glcep,gld,gleps,glp,gp,gphbl,gplbl,gpobl,gprbl,gptbl,gwo,hedgegl,ib,ibadj,ibbl,ibcom,ibki,idiis,idilb,idilc,idis,idist,idit,idits,iire,initb,intc,iobd,ioi,iore,ipabl,iphbl,iplbl,ipobl,iptbl,ipti,irei,irent,irii,irli,irnli,irsi,isgr,isgt,isgu,itci,ivi,li,llrci,llrcr,llwoci,llwocr,lst,mii,nco,nfsr,ni,niadj,nieci,niint,niit,nim,nio,nit,nits,nopi,nopio,nrtxt,nrtxtd,nrtxteps,oiadp,oibdp,opeps,opili,opincar,opini,opioi,opiri,opiti,oprepsx,palr,panlr,patr,pcl,pclr,pcnlr,pctr,pi,pidom,pifo,pll,pltbl,pnca,pncad,pncaeps,pncia,pncid,pncieps,pncip,pncwia,pncwid,pncwieps,pncwip,pnlbl,pnli,pobl,ppcbl,pppabl,ppphbl,pppobl,ppptbl,prca,prcad,prcaeps,prebl,pri,ptbl,ptran,pvo,pvon,pwoi,rca,rcd,rceps,rcp,rdip,rdipa,rdipd,rdipeps,revt,ris,rmum,rra,rrd,rrp,sale,seta,setd,seteps,setp,spce,spced,spceeps,spi,spid,spieps,spioa,spiop,sret,stkco,stkcpa,tdsg,tf,tie,tii,txc,txdfed,txdfo,txdi,txds,txeqa,txeqii,txfed,txfo,txo,txs,txt,txva,txw,udpfa,udvp,ugi,uniami,unopinc,uopi,updvp,uspi,usubdvp,utme,utxfed,uxinst,uxintd,wda,wdd,wdeps,wdp,xad,xago,xagt,xcom,xcomi,xdepl,xdp,xdvre,xeqo,xi,xido,xindb,xindc,xins,xinst,xint,xintd,xintopt,xivi,xivre,xlr,xnbi,xnf,xnins,xnitb,xobd,xoi,xopr,xoprar,xoptd,xopteps,xore,xpr,xrd,xrent,xs,xsga,xstf,xstfo,xstfws,xt,xuw,xuwli,xuwnli,xuwoi,xuwrei,xuwti,afudcc,afudci,amc,aolo

<h3> keep selected cols -> columns

In [107]:
# keep total
keep = pd.read_excel('Variable reference.xlsx')

In [109]:
keep

,Unnamed: 0,columns,variables name,Unnamed: 3
0,1,aco,(aco) Current Assets - Other - Total,
1,2,acoxar,(acoxar) Current Assets - Other - Total As Rep...,NaN
2,3,act,(act) Current Assets - Total,NaN
3,4,artfs,(artfs) Accounts Receivable/Debtors - Total,NaN
4,5,at,(at) Assets - Total,NaN
...,...,...,...,...
102,103,fuset,(fuset) Uses of Funds - Total,NaN
103,104,tsafc,(tsafc) Total Sources/Applications of Funds (C...,NaN
104,105,utfdoc,(utfdoc) Total Funds From Operations (Cash Flow),NaN
105,106,utfosc,(utfosc) Total Funds from Outside Sources (Cas...,NaN


In [111]:
fin = fin[['gvkey', 'datadate', 'sic', 'costat', 'tic', 'conm'] + keep['columns'].tolist()].copy()

## Merge bkpy and fin

### gvkey一致 PERMNO不一致

In [113]:
# check duplicate gvkey
print(bkpy[bkpy['gvkey'].duplicated(keep=False)])

    PERMNO    gvkey delist date  delist year
14   14011  18086.0  2020-10-09         2020
68   23497  18086.0  2023-08-25         2023
80   62368   8515.0  2019-03-25         2019
81   62376   8515.0  2019-03-25         2019


In [115]:
print(merge_delist_link.loc[merge_delist_link['gvkey'] ==  18086, ['ticker', 'Company Name']])
print(merge_delist_link.loc[merge_delist_link['gvkey'] ==  8515, ['ticker', 'Company Name']])

   ticker      Company Name
14  MNKTQ  MALLINCKRODT PLC
68  MNKTQ  MALLINCKRODT PLC
   ticker Company Name
80   PHIG  PHI GRP INC
81   PHIG  PHI GRP INC


gvkey:18086 -> keep

2020&2023 Mallinckrodt 提交chapter 11 破產申請，並退出New York Stock Exchange (NYSE)
*重新在 NYSE American 上市，重新分配新的PERMNO（上市板塊變化，因此PERMNO不一致）


gvkey:8515 -> drop 1

2019 PHI 提交chapter 11 破產申請，並退出Nasdaq
公司公告其股票預計從2019年3月26日起在 OTC 市場（Pink Sheets/OTC）以代號 “PHIIDQ” 與 “PHIIKDQ” 交易
（從場內轉為場外交易，取消原股票，新增新股票，因此重新分配新的PERMNO）
\* 但只有permno不一致，其餘均一致，該筆取一筆 \*

In [117]:
bkpy = bkpy.drop(81)
bkpy = bkpy.reset_index(drop=True)

In [119]:
bkpy.shape

(110, 4)

### PERMNO一致 gvkey不一致 （併購、反向合併、重組、或公司名稱變更事件）

In [121]:
# check duplicate PERMNO
print(bkpy[bkpy['PERMNO'].duplicated(keep=False)])

    PERMNO     gvkey delist date  delist year
9    13263  190580.0  2020-01-10         2020
10   13263  196856.0  2020-01-10         2020
18   14756  317260.0  2020-09-23         2020
19   14756  335466.0  2020-09-23         2020
25   15443   19722.0  2023-11-28         2023
26   15443   36506.0  2023-11-28         2023
30   15970   26825.0  2020-07-31         2020
31   15970   31156.0  2020-07-31         2020
39   17090   32561.0  2023-12-26         2023
40   17090   37426.0  2023-12-26         2023
49   19539   36418.0  2024-02-02         2024
50   19539   38949.0  2024-02-02         2024
52   19950   33433.0  2023-11-03         2023
53   19950   36781.0  2023-11-03         2023
67   51692    3561.0  2020-02-14         2020
68   51692    8579.0  2020-02-14         2020
72   64822    8838.0  2020-04-15         2020
73   64822   21732.0  2020-04-15         2020
74   64822   28788.0  2020-04-15         2020
77   76858   23062.0  2020-05-27         2020
78   76858   26863.0  2020-05-27  

在bkpy裡面PERMNO一致，但是gvkey不一致，如果bkpy僅根據重複值刪除，通常僅保留第一筆，很可能刪除有資料的值。
通過gvkey在fin裡面找到bkpy需要保留的資料

In [123]:
#找出有重複PERMNO的gvkey清單
glist = bkpy[bkpy['PERMNO'].duplicated(keep=False)]['gvkey'].unique().tolist()
#從fin資料篩選出這些公司資訊
r = fin[fin['gvkey'].isin(glist)][['gvkey', 'tic', 'conm']].drop_duplicates().reset_index(drop=True)
#加回PERMNO欄位
r['PERMNO'] = r['gvkey'].map(bkpy.drop_duplicates('gvkey').set_index('gvkey')['PERMNO'])
r

,gvkey,tic,conm,PERMNO
0,8579,PIRRQ,PIER 1 IMPORTS INC/DE,51692
1,19722,BPMX,BIOPHARMX CORP,15443
2,26863,CTRCQ,CENTRIC BRANDS INC,76858
3,28788,YUMAQ,YUMA ENERGY INC,64822
4,31156,ROSEQ,ROSEHILL RESOURCES INC,15970
5,32561,LGC,LEGACY ACQUISITION,17090
6,33433,WEWKQ,WEWORK INC,19950
7,36506,TMBRQ,TREX WIND DOWN INC,15443
8,36781,BOWX,BOWX ACQUISITION CORP,19950
9,37426,IDICQ,PARTS ID INC,17090


In [125]:
# need to drop these rows 找出需要刪除的行
drop = bkpy[bkpy['PERMNO'].duplicated(keep=False) & ~bkpy['gvkey'].isin(r['gvkey'])].copy()

# final clean bkpy 移除這些行，得到最終清洗的bkpy
bkpy = bkpy[~bkpy['gvkey'].isin(drop['gvkey'])].copy()

<h4>
check

In [127]:
print(bkpy[bkpy['PERMNO'].duplicated(keep=False)])
print(bkpy[bkpy['gvkey'].duplicated(keep=False)])

    PERMNO    gvkey delist date  delist year
25   15443  19722.0  2023-11-28         2023
26   15443  36506.0  2023-11-28         2023
39   17090  32561.0  2023-12-26         2023
40   17090  37426.0  2023-12-26         2023
52   19950  33433.0  2023-11-03         2023
53   19950  36781.0  2023-11-03         2023
    PERMNO    gvkey delist date  delist year
13   14011  18086.0  2020-10-09         2020
61   23497  18086.0  2023-08-25         2023


**15443: 19722 在2020年併入36506，36506 在2023年提出Ch11 破產申請** 。
**17090: 32561 在2020年併入37426，37426 在2023年提出Ch11 破產申請** 。
**19950: 36781 在2021年併入33433，33433 在2023年提出Ch11 破產申請** 。

19722,32561,36781 delist 標識須mark0

<h3> Merge

In [137]:
merge = pd.merge(fin, bkpy, how='left', on=['gvkey'])

In [139]:
merge.shape

(65674, 116)

# Set dummy variables

### dummy_delist: Find the closest financial report from the delisting date then mark 1, otherwise 0

In [141]:
merge['datadate'] = pd.to_datetime(merge['datadate'], errors='coerce')
merge['delist date'] = pd.to_datetime(merge['delist date'], errors='coerce')
merge['year'] = merge['datadate'].dt.year

merge = merge.sort_values(["gvkey", "delist date"])

In [143]:
merge.shape

(65674, 117)

In [145]:
# 移除沒有 Delisting Date 的公司
valid = merge.dropna(subset=['delist date']).copy()
# 計算距離（天數差）
valid['date_diff'] = (valid['delist date'] - valid['datadate']).abs()
# 找出每家公司距離最接近破產日的那筆 datadate
idx_min = valid.groupby('gvkey')['date_diff'].idxmin()
closest = valid.loc[idx_min].copy()

# if datadate < delist date -> use current year, else: use last year
closest['target_year'] = np.where(
    closest['datadate'] < closest['delist date'],
    closest['year'],
    closest['year'] - 1
)
# 初始化 dummy_delist 欄位為 0
merge['dummy_delist'] = 0

In [147]:
for _, row in closest.iterrows():
    gv = row['gvkey']
    ty = row['target_year']
    merge.loc[(merge['gvkey'] == gv) & (merge['year'] == ty), 'dummy_delist'] = 1


dummy_delist: 19722,32561,36781 mark 0

In [149]:
merge.loc[merge['gvkey'].isin([19722, 32561, 36781]), 'dummy_delist'] = 0

In [151]:
merge.shape

(65674, 118)

In [153]:
merge.head()

,gvkey,datadate,sic,costat,tic,conm,aco,acoxar,act,artfs,at,ceq,clt,dlc,dltt,dptb,dptc,esopct,esopt,iali,ialti,iasti,iatci,iati,iatmi,iaui,icapt,intan,invt,islt,ist,istc,istm,isut,ivpt,ivst,lcabg,lcat,lco,lcoxar,lct,lcuacu,llot,lo,lse,lt,mrct,nat,npat,ppegt,ppent,prvt,pstk,pvt,rati,rect,ret,rlt,rvbti,rvdt,rveqt,rvnt,rvti,srt,tdst,tstk,txndb,uceq,batr,bcltbl,bct,bctr,bltbl,cgti,citotal,cnltbl,dvt,gptbl,idit,initb,iptbl,ipti,isgt,ivi,nit,nits,opiti,patr,pctr,pltbl,pnlbl,ppptbl,ptbl,revt,tie,tii,txt,uopi,utme,xagt,xint,xlr,xnitb,xopr,xt,xuwti,fopt,fsrct,fuset,tsafc,utfdoc,utfosc,wcapch,PERMNO,delist date,delist year,year,dummy_delist
0,1004,2018-05-31,5080,A,AIR,AAR CORP,150.2,NaN,942.7,NaN,1524.7,936.3,NaN,0.0,177.2,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1113.5,157.1,547.9,NaN,NaN,NaN,NaN,NaN,NaN,10.5,NaN,NaN,163.3,NaN,333.3,NaN,NaN,62.2,1524.7,588.4,67.6,NaN,NaN,531.0,316.6,NaN,0.0,NaN,NaN,203.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,280.7,-15.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23.5,NaN,10.3,NaN,0.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1748.3,NaN,NaN,3.5,NaN,NaN,NaN,8.0,NaN,NaN,1621.8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,2018,0
1,1004,2019-05-31,5080,A,AIR,AAR CORP,64.3,NaN,952.5,NaN,1517.2,905.9,NaN,0.0,141.7,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1047.6,145.6,589.0,NaN,NaN,NaN,NaN,NaN,NaN,19.8,NaN,NaN,169.7,NaN,357.5,NaN,NaN,112.1,1517.2,611.3,81.6,NaN,NaN,580.6,348.8,NaN,0.0,NaN,NaN,258.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,287.7,3.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.4,NaN,10.5,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2051.8,NaN,NaN,4.9,NaN,NaN,NaN,9.5,NaN,NaN,1898.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,2019,0
2,1004,2020-05-31,5080,A,AIR,AAR CORP,92.2,NaN,1438.7,NaN,2079.0,902.6,NaN,13.7,670.9,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1573.5,125.8,692.7,NaN,NaN,NaN,NaN,NaN,NaN,20.0,NaN,NaN,177.8,NaN,383.1,NaN,NaN,122.4,2079.0,1176.4,62.7,NaN,NaN,683.6,437.1,NaN,0.0,NaN,NaN,229.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,282.7,3.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.7,NaN,10.7,NaN,0.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2089.3,NaN,NaN,5.6,NaN,NaN,NaN,9.3,NaN,NaN,1939.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,2020,0
3,1004,2021-05-31,5080,A,AIR,AAR CORP,47.2,NaN,937.0,NaN,1539.7,974.4,NaN,11.5,193.6,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1168.0,148.8,591.0,NaN,NaN,NaN,NaN,NaN,NaN,8.4,NaN,NaN,197.4,NaN,336.8,NaN,NaN,25.4,1539.7,565.3,52.0,NaN,NaN,640.3,380.1,NaN,0.0,NaN,NaN,238.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,274.1,-9.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,62.1,NaN,0.1,NaN,0.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1651.4,NaN,NaN,18.2,NaN,NaN,NaN,5.0,NaN,NaN,1549.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,2021,0
4,1004,2022-05-31,5080,A,AIR,AAR CORP,53.9,NaN,1007.2,NaN,1573.9,1034.5,NaN,11.1,156.3,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1190.8,141.9,604.1,NaN,NaN,NaN,NaN,NaN,NaN,5.4,NaN,NaN,180.7,NaN,348.2,NaN,NaN,14.9,1573.9,539.4,54.7,NaN,NaN,607.5,349.2,NaN,0.0,NaN,NaN,290.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,289.1,-20.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,77.4,NaN,0.0,NaN,0.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1817.1,NaN,NaN,26.6,NaN,NaN,NaN,2.4,NaN,NaN,1667.8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,2022,0


### dummy_delistconm: mark 1 if the company occured delist

In [155]:
merge['dummy_delistconm'] = merge['gvkey'].isin(merge.loc[merge['dummy_delist'] == 1, 'gvkey']).astype(int)

In [157]:
merge.shape

(65674, 119)

In [159]:
merge.head()

,gvkey,datadate,sic,costat,tic,conm,aco,acoxar,act,artfs,at,ceq,clt,dlc,dltt,dptb,dptc,esopct,esopt,iali,ialti,iasti,iatci,iati,iatmi,iaui,icapt,intan,invt,islt,ist,istc,istm,isut,ivpt,ivst,lcabg,lcat,lco,lcoxar,lct,lcuacu,llot,lo,lse,lt,mrct,nat,npat,ppegt,ppent,prvt,pstk,pvt,rati,rect,ret,rlt,rvbti,rvdt,rveqt,rvnt,rvti,srt,tdst,tstk,txndb,uceq,batr,bcltbl,bct,bctr,bltbl,cgti,citotal,cnltbl,dvt,gptbl,idit,initb,iptbl,ipti,isgt,ivi,nit,nits,opiti,patr,pctr,pltbl,pnlbl,ppptbl,ptbl,revt,tie,tii,txt,uopi,utme,xagt,xint,xlr,xnitb,xopr,xt,xuwti,fopt,fsrct,fuset,tsafc,utfdoc,utfosc,wcapch,PERMNO,delist date,delist year,year,dummy_delist,dummy_delistconm
0,1004,2018-05-31,5080,A,AIR,AAR CORP,150.2,NaN,942.7,NaN,1524.7,936.3,NaN,0.0,177.2,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1113.5,157.1,547.9,NaN,NaN,NaN,NaN,NaN,NaN,10.5,NaN,NaN,163.3,NaN,333.3,NaN,NaN,62.2,1524.7,588.4,67.6,NaN,NaN,531.0,316.6,NaN,0.0,NaN,NaN,203.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,280.7,-15.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23.5,NaN,10.3,NaN,0.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1748.3,NaN,NaN,3.5,NaN,NaN,NaN,8.0,NaN,NaN,1621.8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,2018,0,0
1,1004,2019-05-31,5080,A,AIR,AAR CORP,64.3,NaN,952.5,NaN,1517.2,905.9,NaN,0.0,141.7,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1047.6,145.6,589.0,NaN,NaN,NaN,NaN,NaN,NaN,19.8,NaN,NaN,169.7,NaN,357.5,NaN,NaN,112.1,1517.2,611.3,81.6,NaN,NaN,580.6,348.8,NaN,0.0,NaN,NaN,258.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,287.7,3.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.4,NaN,10.5,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2051.8,NaN,NaN,4.9,NaN,NaN,NaN,9.5,NaN,NaN,1898.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,2019,0,0
2,1004,2020-05-31,5080,A,AIR,AAR CORP,92.2,NaN,1438.7,NaN,2079.0,902.6,NaN,13.7,670.9,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1573.5,125.8,692.7,NaN,NaN,NaN,NaN,NaN,NaN,20.0,NaN,NaN,177.8,NaN,383.1,NaN,NaN,122.4,2079.0,1176.4,62.7,NaN,NaN,683.6,437.1,NaN,0.0,NaN,NaN,229.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,282.7,3.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.7,NaN,10.7,NaN,0.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2089.3,NaN,NaN,5.6,NaN,NaN,NaN,9.3,NaN,NaN,1939.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,2020,0,0
3,1004,2021-05-31,5080,A,AIR,AAR CORP,47.2,NaN,937.0,NaN,1539.7,974.4,NaN,11.5,193.6,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1168.0,148.8,591.0,NaN,NaN,NaN,NaN,NaN,NaN,8.4,NaN,NaN,197.4,NaN,336.8,NaN,NaN,25.4,1539.7,565.3,52.0,NaN,NaN,640.3,380.1,NaN,0.0,NaN,NaN,238.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,274.1,-9.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,62.1,NaN,0.1,NaN,0.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1651.4,NaN,NaN,18.2,NaN,NaN,NaN,5.0,NaN,NaN,1549.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,2021,0,0
4,1004,2022-05-31,5080,A,AIR,AAR CORP,53.9,NaN,1007.2,NaN,1573.9,1034.5,NaN,11.1,156.3,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1190.8,141.9,604.1,NaN,NaN,NaN,NaN,NaN,NaN,5.4,NaN,NaN,180.7,NaN,348.2,NaN,NaN,14.9,1573.9,539.4,54.7,NaN,NaN,607.5,349.2,NaN,0.0,NaN,NaN,290.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,289.1,-20.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,77.4,NaN,0.0,NaN,0.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1817.1,NaN,NaN,26.6,NaN,NaN,NaN,2.4,NaN,NaN,1667.8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,2022,0,0


In [161]:
merge.loc[merge['gvkey'].isin([19722, 32561, 36781]), 'dummy_delistconm'] = 1

In [163]:
merge.shape

(65674, 119)

### dummy_costat

In [165]:
merge = merge.rename(columns={'costat': 'dummy_costat'})
merge['dummy_costat'] = merge['dummy_costat'].astype(str).replace({'A': 0, 'I': 1}).astype("Int64")

/var/folders/36/hd8g5yqj6dnflvczdcddf9900000gn/T/ipykernel_25936/530900466.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  merge['dummy_costat'] = merge['dummy_costat'].astype(str).replace({'A': 0, 'I': 1}).astype("Int64")


In [167]:
merge.shape

(65674, 119)

In [169]:
merge.head()

,gvkey,datadate,sic,dummy_costat,tic,conm,aco,acoxar,act,artfs,at,ceq,clt,dlc,dltt,dptb,dptc,esopct,esopt,iali,ialti,iasti,iatci,iati,iatmi,iaui,icapt,intan,invt,islt,ist,istc,istm,isut,ivpt,ivst,lcabg,lcat,lco,lcoxar,lct,lcuacu,llot,lo,lse,lt,mrct,nat,npat,ppegt,ppent,prvt,pstk,pvt,rati,rect,ret,rlt,rvbti,rvdt,rveqt,rvnt,rvti,srt,tdst,tstk,txndb,uceq,batr,bcltbl,bct,bctr,bltbl,cgti,citotal,cnltbl,dvt,gptbl,idit,initb,iptbl,ipti,isgt,ivi,nit,nits,opiti,patr,pctr,pltbl,pnlbl,ppptbl,ptbl,revt,tie,tii,txt,uopi,utme,xagt,xint,xlr,xnitb,xopr,xt,xuwti,fopt,fsrct,fuset,tsafc,utfdoc,utfosc,wcapch,PERMNO,delist date,delist year,year,dummy_delist,dummy_delistconm
0,1004,2018-05-31,5080,0,AIR,AAR CORP,150.2,NaN,942.7,NaN,1524.7,936.3,NaN,0.0,177.2,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1113.5,157.1,547.9,NaN,NaN,NaN,NaN,NaN,NaN,10.5,NaN,NaN,163.3,NaN,333.3,NaN,NaN,62.2,1524.7,588.4,67.6,NaN,NaN,531.0,316.6,NaN,0.0,NaN,NaN,203.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,280.7,-15.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23.5,NaN,10.3,NaN,0.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1748.3,NaN,NaN,3.5,NaN,NaN,NaN,8.0,NaN,NaN,1621.8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,2018,0,0
1,1004,2019-05-31,5080,0,AIR,AAR CORP,64.3,NaN,952.5,NaN,1517.2,905.9,NaN,0.0,141.7,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1047.6,145.6,589.0,NaN,NaN,NaN,NaN,NaN,NaN,19.8,NaN,NaN,169.7,NaN,357.5,NaN,NaN,112.1,1517.2,611.3,81.6,NaN,NaN,580.6,348.8,NaN,0.0,NaN,NaN,258.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,287.7,3.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.4,NaN,10.5,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2051.8,NaN,NaN,4.9,NaN,NaN,NaN,9.5,NaN,NaN,1898.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,2019,0,0
2,1004,2020-05-31,5080,0,AIR,AAR CORP,92.2,NaN,1438.7,NaN,2079.0,902.6,NaN,13.7,670.9,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1573.5,125.8,692.7,NaN,NaN,NaN,NaN,NaN,NaN,20.0,NaN,NaN,177.8,NaN,383.1,NaN,NaN,122.4,2079.0,1176.4,62.7,NaN,NaN,683.6,437.1,NaN,0.0,NaN,NaN,229.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,282.7,3.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.7,NaN,10.7,NaN,0.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2089.3,NaN,NaN,5.6,NaN,NaN,NaN,9.3,NaN,NaN,1939.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,2020,0,0
3,1004,2021-05-31,5080,0,AIR,AAR CORP,47.2,NaN,937.0,NaN,1539.7,974.4,NaN,11.5,193.6,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1168.0,148.8,591.0,NaN,NaN,NaN,NaN,NaN,NaN,8.4,NaN,NaN,197.4,NaN,336.8,NaN,NaN,25.4,1539.7,565.3,52.0,NaN,NaN,640.3,380.1,NaN,0.0,NaN,NaN,238.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,274.1,-9.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,62.1,NaN,0.1,NaN,0.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1651.4,NaN,NaN,18.2,NaN,NaN,NaN,5.0,NaN,NaN,1549.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,2021,0,0
4,1004,2022-05-31,5080,0,AIR,AAR CORP,53.9,NaN,1007.2,NaN,1573.9,1034.5,NaN,11.1,156.3,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1190.8,141.9,604.1,NaN,NaN,NaN,NaN,NaN,NaN,5.4,NaN,NaN,180.7,NaN,348.2,NaN,NaN,14.9,1573.9,539.4,54.7,NaN,NaN,607.5,349.2,NaN,0.0,NaN,NaN,290.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,289.1,-20.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,77.4,NaN,0.0,NaN,0.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1817.1,NaN,NaN,26.6,NaN,NaN,NaN,2.4,NaN,NaN,1667.8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,2022,0,0


# sic_rank: class company by sic

In [171]:
# 計算大分類 industry
merge['sic_ind']= (merge['sic'] // 100) * 100

In [173]:
merge.shape

(65674, 120)

In [175]:
merge['sic_rank'] = pd.factorize(merge['sic_ind'])[0] + 1 #將類別變數轉換成數字 1最小，方便模型處理。

In [177]:
merge.shape

(65674, 121)

# dummy_sic

In [179]:
merge = pd.get_dummies(merge, columns=['sic_ind'], prefix='dummy', dtype=int)

In [181]:
merge.shape

(65674, 191)

# missing indicator(merge1)

In [183]:
# del 100% missing cols
#計算每個欄位的缺失比例（NaN比例）
missing_ratio = merge.isnull().mean().sort_values(ascending=False)
#找出所有缺失率等於1（100%缺失）的欄位名稱：
missing100_cols = missing_ratio[missing_ratio == 1.0].index
#將這些完全缺失的欄位刪除：
merge_mi = merge.drop(columns=missing100_cols)

In [185]:
merge_mi.shape

(65674, 186)

In [187]:
def add_missing_indicators(df):
    """
    為每個含有缺值的欄位新增 dummy 欄位
    命名格式：dummy_{col}_missing
    """
    df_out = df.copy()

    # 🔹 使用顯式判斷，避免 Series truth value 錯誤
    cols_with_missing = [col for col in df.columns if df[col].isna().any().item()]

    # 🔹 建立指示變數 DataFrame
    indicators = {
        f"dummy_{col}_missing": df[col].isna().astype(int)
        for col in cols_with_missing
    }
    indicators_df = pd.DataFrame(indicators, index=df.index)

    # 🔹 合併
    df_out = pd.concat([df_out, indicators_df], axis=1)

    return df_out

In [189]:
merge1 = add_missing_indicators(merge_mi)

In [191]:
merge1.shape

(65674, 291)

# Log(merge2)

In [193]:
def log_transform(df, dummy_keywords=["dummy"], exclude_cols=None, inplace=False):
    """
    對除指定欄位以外的數值欄位進行 log 轉換：
    - 若最小值 <=0 → 平移使所有值 >0，再取 log(x_shifted)
    - 自動跳過 NaN，但保留為 NaN
    - 避免對 dummy 欄位及 exclude_cols 指定欄位進行轉換

    參數：
        df : DataFrame
        dummy_keywords : list[str]，dummy 欄位名稱中包含的關鍵字
        exclude_cols : list[str]，要排除不取 log 的欄位名稱
        inplace : bool，是否直接修改原始 df

    回傳：
        df_out : 轉換後的 DataFrame
        log_cols : 實際做 log 轉換的欄位清單
    """
    if not inplace:
        df = df.copy()

    if exclude_cols is None:
        exclude_cols = []

    # 取得數值欄位
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    # 排除 dummy 欄位與指定排除欄位
    non_dummy_cols = [
        col for col in numeric_cols
        if not any(key.lower() in col.lower() for key in dummy_keywords)
        and col not in exclude_cols
    ]

    log_cols = []

    for col in non_dummy_cols:
        series = df[col]

        if series.isna().all():
            df[col] = np.nan
            continue

        # 若最小值 <= 0，平移使所有值 > 0
        min_val = series.min(skipna=True)
        if min_val < 0:
            # 若有負值 → 平移到全為正後取 log
            shift = abs(min_val) + 1e-6
            transformed = np.log(series + shift)
        else:
            # 若最小值 ≥ 0，包含 0 → 使用 log1p(x)
            transformed = np.log1p(series)

        df[col] = np.where(series.notna(), transformed, np.nan)
        log_cols.append(col)

    print(f"已完成所有 {len(log_cols)} 個欄位的 log 轉換。")
    return df, log_cols


In [195]:
exclude_list = ['gvkey', 'datadate', 'sic', 'PERMNO', 'delist date', 'delist year', 'tic', 'conm', 'year','sic_rank']
merge2, transformed_cols = log_transform(merge1, exclude_cols=exclude_list)

已完成所有 102 個欄位的 log 轉換。


# range year(result)

In [197]:
# from 2019 to 2024
result = merge2[(merge2['year'] >= 2019) & (merge2['year'] <= 2024)].copy()

# OUTPUT

In [199]:
result.to_csv('clean_data.csv', index=False)